# Diagnosing a Reconstructor CNN with `CNNDiagnostics`

> **Where this fits.** This notebook diagnoses the network trained in `basics/05_TrainingAReconstructor.ipynb`. Unlike the other advanced notebooks, it uses that notebook's params file (`../../basics/wfs_params_exp.py`) instead of one of its own, so the architecture and the twin match the trained weights exactly.

`basics/05_TrainingAReconstructor.ipynb` trained a CNN reconstructor end-to-end through the WFS → DM simulation, but only showed its *output*: the loss curve and the corrected wavefront. This notebook looks *inside* the network with `AI4AO.CNNDiagnostics` (design: `Ideas/17-cnn-diagnostics.md`):

- what the intermediate **feature maps** and the learned **kernels** look like;
- how the **gradients flow** from the loss back to the first layer, and whether some layers get too little or too much of it;
- which **pupil pixels** the loss (saliency) and each predicted mode (Jacobian) are sensitive to;
- how all of this **changes along a closed-loop rollout** and **over training steps**.

The key idea: `CNNDiagnostics` never runs its own training. It attaches hooks around *your* forward/backward pass (`with diag.capture(): ...`), so the gradients it shows are the real ones, coming through the whole differentiable WFS/DM chain.

**Prerequisite:** run `basics/05_TrainingAReconstructor.ipynb` first so `Data/Tutorials/ReconstructorCNN.pth` exists. Without it, the "trained" network below is just randomly initialized, and the plots still work but show an untrained network.

## Setup

Same configuration, dataset, WFS, DM, preprocessor and `PWFSNet` architecture as `basics/05_TrainingAReconstructor.ipynb`. The architecture must match exactly for the trained weights to load.

In [ ]:
from contextlib import nullcontext

from mmengine import Config
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from IPython.display import display

from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, CNNDiagnostics
from AI4AO.paths import DATA_DIR
from AI4AO.LossFunctions import LogResidualVarianceLoss, Physics_loss

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# The params file of basics/05_TrainingAReconstructor.ipynb, whose trained network this
# notebook diagnoses: the twin and the architecture must match its checkpoint.
PARAMFILE = "../../basics/wfs_params_exp.py"

AtmosParams = Config.fromfile(PARAMFILE)['AtmosParams']
WFSParams = Config.fromfile(PARAMFILE)['WFSParams']
LoopParams = Config.fromfile(PARAMFILE)['LoopParams']
TrainParams = Config.fromfile(PARAMFILE)['TrainParams']
DMParams = Config.fromfile(PARAMFILE)['DMParams']

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

In [ ]:
PATH = DATA_DIR / "Tutorials"

wfs = PyramidWFS(WFSParams, device)
dm = DeformableMirror(WFSParams, DMParams, device)

try:
    wfs.LoadCalibration(PATH / "TutorialsWFS.pth")
except FileNotFoundError:
    print("No saved WFS calibration found -- using the uncalibrated WFS")

# The diagnostics below do not depend on the twin being calibrated, so a DM
# calibration that cannot be loaded only costs realism, not the demo.
try:
    dm.LoadCalibration(PATH / "TutorialsDM.pth")
except (FileNotFoundError, RuntimeError) as err:
    print(f"DM calibration not loaded ({type(err).__name__}: {str(err).splitlines()[0]}) "
          "-- using the uncalibrated DM")

wfs.eval()
dm.eval()

framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

M2C = dm.MakeZernikeM2C()
z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))

In [ ]:
class PWFSNet(nn.Module):
    def __init__(self, DMParams):
        super().__init__()

        Nmodes = DMParams["Nmodes"]

        self.stem = nn.Sequential(
            # Process each pupil independently
            nn.Conv2d(4, 32, kernel_size=11, padding=5, groups=4),
            nn.GELU(),

            nn.Conv2d(32, 64, kernel_size=7, padding=3, groups=4),
            nn.GELU(),

            nn.MaxPool2d(2),
        )

        self.encoder = nn.Sequential(
            nn.Conv2d(64, 64, 5, padding=2),
            nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, 2, padding=1),
            nn.GELU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, Nmodes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.encoder(x)
        return self.head(x)


CHECKPOINT_PATH = DATA_DIR / "Tutorials" / "ReconstructorCNN.pth"

phaseReconstructor = PWFSNet(DMParams).to(device)
if CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    phaseReconstructor.load_state_dict(checkpoint["phase_reconstructor_state_dict"])
    print(f"Loaded trained reconstructor from {CHECKPOINT_PATH}")
else:
    print("No trained checkpoint found -- diagnosing a randomly initialized network")

## One closed-loop training step, written out

`CNNDiagnostics` observes whatever forward/backward pass runs inside its `capture()` block. To give it a realistic one, `closed_loop_step` below is a copy of the inner loop of `Trainer.train` (`AI4AO/Trainer.py`): draw a batch of turbulence, run `closed_loop_iterations` AO-loop steps (WFS → preprocessing → CNN → DM), and backpropagate the same loss as `basics/05` into the network.

With `optimizer=None` it only computes gradients and leaves the weights unchanged, which is what we want for inspecting the trained network. It returns the loss and the preprocessed WFS frames of the last loop iteration (the CNN's input), which section 8 reuses.

In [ ]:
loss_fn = LogResidualVarianceLoss(dataset.pupil, wfs.wavelength) + Physics_loss(wfs=wfs)


def closed_loop_step(model, closed_loop_iterations=1, optimizer=None):
    with torch.no_grad():
        batch = dataset[0]
        opd_gt, pupil_gt = batch["opd"], batch["pupil"]
        gain, leak = batch["loop_gain"], batch["loop_leak"]
        wfs.SetPhotonsAndRON(batch["nphotons"], batch["ron"])

        modes = opd_gt.flatten(start_dim=-2) @ z_inv
        z_estimated = torch.zeros_like(modes)
        z_buffer = torch.zeros_like(modes)
        z_output = torch.zeros_like(modes)
        opd_reconstructed = torch.zeros_like(opd_gt)

    total_loss = 0
    for i in range(closed_loop_iterations):
        with torch.no_grad():
            if i > 0:
                batch = dataset[i]
                opd_gt, pupil_gt = batch["opd"], batch["pupil"]

            residual_opd = opd_gt - opd_reconstructed
            Ze = residual_opd.flatten(start_dim=-2) @ z_inv

            z_estimated = z_estimated * leak + gain * z_buffer
            z_buffer = z_output.detach().clone()

            wfs_frames = wfs(residual_opd, pupil_gt)
            frames = framePreprocessor.ProcessFrame(wfs_frames)

        z_output = model(frames)

        opd_reconstructed = dm(z_estimated @ M2C.T)
        corrected_residual_opd = residual_opd - dm(z_output @ M2C.T)
        total_loss = total_loss + loss_fn(
            Ze, z_output, pupil_gt, residual_opd, corrected_residual_opd, wfs_frames
        ) / closed_loop_iterations

    model.zero_grad(set_to_none=True)
    total_loss.backward()
    if optimizer is not None:
        optimizer.step()

    return total_loss.detach(), frames

## 1. Which layers are hooked

By default `CNNDiagnostics` hooks every `Conv2d` and `Linear` layer, named by its `named_modules()` path. Every method also accepts an integer index into `diag.layers`. `max_samples` caps how many batch samples are *stored* for plotting; the scalar statistics always use the full batch.

`PWFSNet` uses `GELU` modules, so the hooked conv outputs are *pre-activation* maps. To also see the post-GELU maps, pass `layer_types=(nn.Conv2d, nn.Linear, nn.GELU)`.

In [ ]:
diag = CNNDiagnostics(phaseReconstructor, max_samples=4)
list(diag.layers)

## 2. Capture one real training step

Everything the network sees and receives during one closed-loop step is recorded inside the `with` block. `input_grad=True` additionally records the gradient of the loss with respect to the network's input (the preprocessed pupils). The hooks are removed when the block ends, so later code runs untouched.

`diag.record(step)` turns the capture into one row of per-layer scalars. `diag.summary()` prints the latest row:

- `weight_grad_norm`: ‖∂L/∂W‖ of the layer's weights.
- `act_grad_rms`: RMS of the gradient reaching the layer's output.
- `update_ratio`: ‖ΔW‖/‖W‖ over the optimizer step. It is 0 here because no step is taken.
- `dead_frac`: fraction of channels whose pre-activation output is ≤ 0 for every sample and pixel. With GELU this is only indicative: such channels are nearly, not exactly, silent. It is not defined (`nan`) for `head.1`, whose output is the network's output and feeds no nonlinearity.

In [ ]:
phaseReconstructor.train()

with diag.capture(input_grad=True):
    loss, frames = closed_loop_step(phaseReconstructor)

diag.record(0)
print(f"loss = {float(loss):.3f}   CNN input: {tuple(frames.shape)}")
print(diag.summary())

## 3. Feature maps

Feature maps are the layer outputs, shown as one boxed group per sample. `stem.0` is a grouped convolution (`groups=4`): channels 0–7 only see pupil 0, channels 8–15 only pupil 1, and so on.

For layers with many channels, `mosaic=True` tiles everything into one image, which is much faster to draw: one block of channels per sample, separated by gray gaps. The last layer, `head.1`, is the `Linear` output, i.e. the predicted mode coefficients, drawn as one line per sample.

In [ ]:
diag.plot_feature_maps("stem.0", max_samples=2, max_channels=16);

In [ ]:
diag.plot_feature_maps("encoder.3", max_samples=2, max_channels=32, mosaic=True);
diag.plot_feature_maps("head.1");

## 4. Kernels and their gradients

`plot_kernels` shows a conv layer's weights on a symmetric red/blue scale (white = 0). For the first layer these are the 11×11 filters applied directly to the pupil images. `grad=True` shows the gradient the captured step pushed into the same weights.

In [ ]:
diag.plot_kernels("stem.0", max_out=32, mosaic=True);
diag.plot_kernels("stem.0", grad=True, max_out=32, mosaic=True);

## 5. Gradient flow

The classic vanishing/exploding-gradient check: mean and max |grad| per layer, from the input (left) to the output (right), on a log scale.

- `source="weight"` uses the weight gradients (what the optimizer sees).
- `source="activation"` uses the gradient reaching each layer's *output*, i.e. the training signal as it travels back through the network.

A healthy network has no layer several orders of magnitude below its neighbours. Adam rescales each parameter's step, so unequal weight gradients are not fatal by themselves, but a layer whose activation gradient collapses is a layer that barely learns.

In [ ]:
diag.plot_gradient_flow(source="weight");
diag.plot_gradient_flow(source="activation");

## 6. Distributions

Small-multiple histograms per layer, shallow (light) to deep (dark). Pre-activation outputs should stay roughly centred with a sensible width; very wide tails or a spike at a single value hint at saturation or dead channels.

In [ ]:
diag.plot_histograms("activation");
diag.plot_histograms("activation_grad");

## 7. Saliency: which pupil pixels drive the loss

`∂L/∂(input)` from the captured step, next to the input frames (4 pupils per sample). Red/blue mark pixels where increasing the intensity would increase/decrease the loss.

In [ ]:
diag.plot_saliency(samples=slice(0, 4));

## 8. Per-mode Jacobians: what each output mode looks at

`diag.jacobian(x, modes)` computes `∂z_k/∂x` for selected output modes `k`, one backward pass per mode (so keep the list short). It switches the network to eval mode for the computation and restores the previous mode afterwards.

`M2C` comes from `dm.MakeZernikeM2C()`, whose mode `i` is the Zernike polynomial with Noll index `i + 2` (piston is excluded). Below, mode 0 is tip, 1 is tilt, 2 is focus, 9 is spherical (Noll 11) and 40 is a high-order mode.

The first figure shows the signed Jacobian for one sample. The second averages |J| over the batch, which shows *where* each mode gets its information regardless of the sign.

In [ ]:
MODES = [0, 1, 2, 9, 40]
x = frames[:8]

J = diag.jacobian(x, MODES)          # (len(MODES), B, 4, H, W)
diag.plot_jacobian(J, x, MODES, sample=0);
diag.plot_jacobian(J.abs().mean(dim=1, keepdim=True), x[:1], MODES, signed=False);

## 9. Along a closed-loop rollout: `calls="all"`

With `closed_loop_iterations > 1`, the CNN runs once per AO-loop step and all of those calls share **one** `backward()`. With `calls="all"`, `capture()` stores one entry per call (`diag.activation_grads[name][i]`), and pairs every gradient with the call that produced it.

Iteration 0 sees the uncorrected turbulence; later iterations see residuals that the loop has already partially corrected. The plot shows how strong the training signal is at each layer for each loop iteration.

In [ ]:
N_LOOP = 5

with diag.capture(calls="all"):
    loss, _ = closed_loop_step(phaseReconstructor, closed_loop_iterations=N_LOOP)

names = diag.forward_order
colors = plt.get_cmap("Blues")(np.linspace(0.35, 1.0, len(names)))

fig, ax = plt.subplots(figsize=(8, 4))
for name, color in zip(names, colors):
    rms = [float(g.pow(2).mean().sqrt()) if g is not None else np.nan
           for g in diag.activation_grads[name]]
    ax.semilogy(range(N_LOOP), rms, "-o", lw=2, ms=5, color=color, label=name)
ax.set_xlabel("closed-loop iteration")
ax.set_ylabel("RMS dL/d(layer output)")
ax.set_xticks(range(N_LOOP))
ax.grid(alpha=0.3)
ax.legend(fontsize="small", loc="center left", bbox_to_anchor=(1.01, 0.5))
fig.tight_layout()
plt.show()

The same data is available for the feature maps; for instance, the first stem layer at the first and at the last loop iteration:

In [ ]:
diag.plot_feature_maps("stem.0", call=0, max_samples=1, max_channels=32, mosaic=True);
diag.plot_feature_maps("stem.0", call=-1, max_samples=1, max_channels=32, mosaic=True);

## 10. Monitoring a training run

The trained network above has already converged. To see the diagnostics *move*, this section trains a fresh, randomly initialized copy for a few steps, capturing every `LOG_EVERY` steps and recording a history row each time.

The gradient-flow figure is updated in place (`fig=`/`axes=` handed back, as with `imshow`) and refreshed through an IPython display handle. Only the fresh copy is trained, and nothing is saved, so the `basics/05` checkpoint is never touched.

`N_STEPS` is deliberately tiny: this is a demonstration of the monitoring loop, not a training run. For a real run, raise it, or copy this loop into your own training script.

In [ ]:
N_STEPS = 60
LOG_EVERY = 5

torch.manual_seed(0)
fresh = PWFSNet(DMParams).to(device)
fresh.train()
optimizer = torch.optim.AdamW(fresh.parameters(), TrainParams['lrn'])
diag_fresh = CNNDiagnostics(fresh)

losses = []
flow_fig = flow_axes = handle = None
for step in range(N_STEPS):
    log = step % LOG_EVERY == 0
    with (diag_fresh.capture() if log else nullcontext()):
        loss, _ = closed_loop_step(fresh, optimizer=optimizer)
    losses.append(float(loss))

    if log:
        diag_fresh.record(step)
        if flow_fig is None:
            flow_fig, flow_axes = diag_fresh.plot_gradient_flow()
            handle = display(flow_fig, display_id=True)
        else:
            diag_fresh.plot_gradient_flow(fig=flow_fig, axes=flow_axes)
            handle.update(flow_fig)

plt.close(flow_fig)  # already displayed through the handle

The recorded history, one line per layer (light = shallow, dark = deep). Things to look for:

- `update_ratio` around 1e-3 per step is the usual healthy range; much smaller means a layer is effectively frozen, much larger means it is being thrown around.
- `weight_grad_norm` should not drift by orders of magnitude, nor collapse for the first layers.
- `dead_frac` creeping up means channels are dying.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(losses, lw=2)
ax.set_xlabel("step")
ax.set_ylabel("training loss")
ax.grid(alpha=0.3)
plt.show()

diag_fresh.plot_history("update_ratio");
diag_fresh.plot_history("weight_grad_norm");
diag_fresh.plot_history("dead_frac");
print(diag_fresh.summary())

## 11. Forward-only inspection, and two rules

For a quick look at the feature maps without any loss, capture a forward pass under `torch.no_grad()` with `activation_grads=False`. Without that flag, `capture()` warns that no gradients arrived, since that usually means a mistake: a forward under `no_grad` (as in `Trainer.evaluate`), or a missing `backward()`.

Two rules when using `capture()` in your own loops:

1. Call `backward()` **inside** the `with` block: parameter gradients are read when the block ends.
2. Don't call `zero_grad()` between `backward()` and the end of the block. `Trainer`'s order (`zero_grad` → `backward` → `step`) is fine.

For a self-contained check that never touches `.grad` (safe to call in the middle of training), use `diag.run(x, loss_fn)`. It uses a CNN-only loss, without the closed loop.

In [ ]:
phaseReconstructor.eval()
with torch.no_grad(), diag.capture(activation_grads=False):
    phaseReconstructor(frames)

diag.plot_feature_maps("encoder.0", max_samples=1, max_channels=64, mosaic=True);

## 12. Trying four fixes

The [PWFSNet structure report](https://claude.ai/artifact/GuWrW3TqjX17TSKMGhFbw5) found four things this section tries to fix:

| Change | Finding it targets | What changes |
|---|---|---|
| **Stem learning rate** | `stem.0`/`stem.2` still look like their random initialization | AdamW parameter group for `model.stem` at `STEM_LR_FACTOR` × `lrn` (try 5–10) |
| **Spatial head** | half the parameters sit in `encoder.9`, which mostly sees padding, and global average pooling discards where features are | drop the last pool and the 2×2 conv; a 1×1 conv squeezes the last map to 64 channels, which is flattened into the linear head |
| **Cross-pupil front** | the grouped stem only combines the four pupils after the first pooling | a learnable 1×1 conv (4 → 4) in front, initialized to the Pyramid combinations: total flux, x-slope, y-slope and the cross term |
| **Ceil pooling** | max-pooling drops the last row and column of odd-sized maps three times | `MaxPool2d(2, ceil_mode=True)` everywhere |

`PWFSNetV2` below holds the three architecture switches; with all of them off it is exactly `PWFSNet`. The stem learning rate is an optimizer setting, made in `make_optimizer`.

Two details:

- With `ceil_pool=True` the maps go 46 → 23 → 12 → 6 instead of 46 → 23 → 11 → 5, so the spatial head keeps a 6×6 map and its linear layer is `Linear(2304, 64)` instead of `Linear(1600, 64)`. The class computes the head size from a dummy forward pass, so any combination works.
- The front conv's initialization is an orthonormal change of basis (a 4×4 Hadamard matrix), so it loses no information; the grouped stem then works on one physical signal per group instead of one pupil per group. The pupil layout is read from `wfs.pupil_centers` rather than assumed.

In [ ]:
import copy

from AI4AO.Utils import fused_optimizer_supported


def pyramid_combinations(pupil_centers):
    # (4, 4) orthonormal map from the four pupils to total flux, x-slope, y-slope and
    # the cross term, built from the pupil layout (`pupil_centers` rows are [y, x]).
    c = torch.as_tensor(np.asarray(pupil_centers), dtype=torch.float32)
    sy = torch.sign(c[:, 0] - c[:, 0].mean())
    sx = torch.sign(c[:, 1] - c[:, 1].mean())
    H = 0.5 * torch.stack([torch.ones_like(sx), sx, sy, sx * sy])
    assert torch.allclose(H @ H.T, torch.eye(4)), "the four pupils are not on a 2x2 grid"
    return H


class PWFSNetV2(nn.Module):
    # PWFSNet with three optional changes (all off = PWFSNet):
    #   cross_pupil  : learnable 1x1 conv in front, initialized to the Pyramid combinations
    #   ceil_pool    : MaxPool2d(2, ceil_mode=True), odd-sized maps keep their last row/column
    #   spatial_head : no last pool / 2x2 conv; a 1x1 conv squeezes the last map to 64
    #                  channels, flattened into the linear head (keeps feature positions)
    def __init__(self, DMParams, input_size, pupil_centers=None,
                 cross_pupil=False, ceil_pool=False, spatial_head=False):
        super().__init__()
        Nmodes = DMParams["Nmodes"]

        def pool():
            return nn.MaxPool2d(2, ceil_mode=ceil_pool)

        self.front = nn.Identity()
        if cross_pupil:
            self.front = nn.Conv2d(4, 4, kernel_size=1)
            with torch.no_grad():
                self.front.weight.copy_(pyramid_combinations(pupil_centers)[:, :, None, None])
                self.front.bias.zero_()

        self.stem = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=11, padding=5, groups=4), nn.GELU(),
            nn.Conv2d(32, 64, kernel_size=7, padding=3, groups=4), nn.GELU(),
            pool(),
        )

        layers = [
            nn.Conv2d(64, 64, 5, padding=2), nn.GELU(), pool(),
            nn.Conv2d(64, 128, 3, padding=1), nn.GELU(), pool(),
            nn.Conv2d(128, 256, 3, padding=1), nn.GELU(),
        ]
        if spatial_head:
            layers += [nn.Conv2d(256, 64, kernel_size=1), nn.GELU()]
        else:
            layers += [pool(), nn.Conv2d(256, 512, 2, padding=1), nn.GELU(), nn.AdaptiveAvgPool2d(1)]
        self.encoder = nn.Sequential(*layers)

        with torch.no_grad():
            probe = torch.zeros(1, 4, input_size, input_size)
            n_features = self.encoder(self.stem(self.front(probe))).numel()
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(n_features, Nmodes))

    def forward(self, x):
        return self.head(self.encoder(self.stem(self.front(x))))


def make_optimizer(model, lr, stem_lr_factor=1.0):
    # AdamW with the stem in its own parameter group at stem_lr_factor x lr.
    stem = list(model.stem.parameters())
    stem_ids = {id(p) for p in stem}
    rest = [p for p in model.parameters() if id(p) not in stem_ids]
    return torch.optim.AdamW(
        [{"params": stem, "lr": lr * stem_lr_factor}, {"params": rest, "lr": lr}],
        fused=fused_optimizer_supported(device),
    )


print("Pyramid combinations (rows: flux, x-slope, y-slope, cross; columns: pupils 0-3):")
print(pyramid_combinations(wfs.pupil_centers))

### The variants

One variant per change, plus the baseline and all four together. Every variant starts from a fresh random initialization with the same seed, so the comparison is between architectures and optimizer settings, not between starting points.

In [ ]:
STEM_LR_FACTOR = 5  # try 5-10
lrn = TrainParams['lrn']
input_size = framePreprocessor.Nout

EXPERIMENTS = {
    "baseline":                     dict(arch={}, stem_lr_factor=1),
    f"stem lr ×{STEM_LR_FACTOR}":   dict(arch={}, stem_lr_factor=STEM_LR_FACTOR),
    "spatial head":                 dict(arch=dict(spatial_head=True), stem_lr_factor=1),
    "cross-pupil front":            dict(arch=dict(cross_pupil=True), stem_lr_factor=1),
    "ceil pooling":                 dict(arch=dict(ceil_pool=True), stem_lr_factor=1),
    "all four":                     dict(arch=dict(spatial_head=True, cross_pupil=True, ceil_pool=True),
                                         stem_lr_factor=STEM_LR_FACTOR),
}

models, optimizers, diags = {}, {}, {}
for name, exp in EXPERIMENTS.items():
    torch.manual_seed(0)
    model = PWFSNetV2(DMParams, input_size, wfs.pupil_centers, **exp["arch"]).to(device)
    model.train()
    models[name] = model
    optimizers[name] = make_optimizer(model, lrn, exp["stem_lr_factor"])
    diags[name] = CNNDiagnostics(model)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{name:<20} {n_params:>10,} parameters   head: {model.head[1]}")

### Training all variants on shared draws

`train_all` trains every variant together. Each step draws the atmosphere **once** and reuses it for every model, together with the first loop iteration's WFS frames (they only depend on the atmosphere, not on the model). Every model then runs its own closed loop, its own backward pass and its own optimizer step, and is captured by its own `CNNDiagnostics` every `LOG_EVERY` steps. `closed_loop_step` from the top of the notebook is left unchanged for single-model use.

Nothing is saved. `N_STEPS` is demonstration length: the curves show the diagnostics moving, not which variant wins. For a real comparison, set `N_STEPS = TrainParams['TrainRunNb']` (5000) and run it yourself; it costs about one `basics/05` training run per variant.

In [ ]:
def variant_loss(model, batches, frames0_raw, frames0):
    # Closed loop for one model on pre-drawn batches; mirrors closed_loop_step.
    gain, leak = batches[0]["loop_gain"], batches[0]["loop_leak"]
    with torch.no_grad():
        opd0 = batches[0]["opd"]
        z_estimated = torch.zeros(opd0.shape[0], z_inv.shape[1], device=opd0.device)
        z_buffer = torch.zeros_like(z_estimated)
        z_output = torch.zeros_like(z_estimated)
        opd_reconstructed = torch.zeros_like(opd0)

    total_loss = 0
    for i, batch in enumerate(batches):
        with torch.no_grad():
            opd_gt, pupil_gt = batch["opd"], batch["pupil"]
            residual_opd = opd_gt - opd_reconstructed
            Ze = residual_opd.flatten(start_dim=-2) @ z_inv

            z_estimated = z_estimated * leak + gain * z_buffer
            z_buffer = z_output.detach().clone()

            if i == 0:  # no correction applied yet: the shared frames are this model's frames
                wfs_frames, frames = frames0_raw, frames0
            else:
                wfs_frames = wfs(residual_opd, pupil_gt)
                frames = framePreprocessor.ProcessFrame(wfs_frames)

        z_output = model(frames)

        opd_reconstructed = dm(z_estimated @ M2C.T)
        corrected_residual_opd = residual_opd - dm(z_output @ M2C.T)
        total_loss = total_loss + loss_fn(
            Ze, z_output, pupil_gt, residual_opd, corrected_residual_opd, wfs_frames
        ) / len(batches)
    return total_loss


def train_all(models, optimizers, diags, n_steps, log_every, closed_loop_iterations=1):
    history = {name: [] for name in models}
    for step in range(n_steps):
        with torch.no_grad():
            batches = [dataset[i] for i in range(closed_loop_iterations)]
            wfs.SetPhotonsAndRON(batches[0]["nphotons"], batches[0]["ron"])
            frames0_raw = wfs(batches[0]["opd"], batches[0]["pupil"])
            frames0 = framePreprocessor.ProcessFrame(frames0_raw)

        log = step % log_every == 0 or step == n_steps - 1
        for name, model in models.items():
            # One model at a time: its graph is freed before the next one is built.
            with (diags[name].capture(activations=False) if log else nullcontext()):
                loss = variant_loss(model, batches, frames0_raw, frames0)
                optimizers[name].zero_grad(set_to_none=True)
                loss.backward()
                optimizers[name].step()
            history[name].append(float(loss.detach()))
            if log:
                diags[name].record(step)

        if log:
            print(f"step {step:>5}  " + "  ".join(f"{n}: {h[-1]:.3f}" for n, h in history.items()))
    return history


N_STEPS = 100
LOG_EVERY = 10

loss_history = train_all(models, optimizers, diags, N_STEPS, LOG_EVERY)

### Results

Fixed colours per variant (baseline in gray) so every plot reads the same way.

- **Loss**: smoothed training loss per variant.
- **Stem update ratio**: did the stem learning rate (and the other changes) make the stem move? The encoder's typical level, about 2×10⁻³, is the target.
- **Per-mode error**: relative error ⟨(ẑ − z)²⟩ / ⟨z²⟩ on fresh batches, in eval mode, without pupil-position augmentation. 1.0 means no better than no correction.

In [ ]:
COLORS = dict(zip(EXPERIMENTS, ["0.45"] + [f"C{i}" for i in range(len(EXPERIMENTS) - 1)]))


def smooth(values, window=10):
    v = np.asarray(values, dtype=float)
    if len(v) < window:
        return v
    return np.convolve(v, np.ones(window) / window, mode="valid")


fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name in EXPERIMENTS:
    axes[0].plot(smooth(loss_history[name]), lw=2, color=COLORS[name], label=name)
    d = diags[name]
    axes[1].semilogy(d.history_steps, d.history["update_ratio"]["stem.0"], "-o", ms=4, lw=2,
                     color=COLORS[name], label=name)
axes[0].set_xlabel("step")
axes[0].set_ylabel("training loss (smoothed)")
axes[1].axhline(2e-3, color="0.3", lw=1, ls=":")
axes[1].set_xlabel("step")
axes[1].set_ylabel("stem.0 update ratio")
for ax in axes:
    ax.grid(alpha=0.3)
axes[1].legend(fontsize="small", loc="center left", bbox_to_anchor=(1.01, 0.5))
fig.tight_layout()
plt.show()

In [ ]:
def per_mode_error(models, n_batches=12):
    for m in models.values():
        m.eval()
    num = {name: 0 for name in models}
    den = 0
    with torch.no_grad():
        for _ in range(n_batches):
            batch = dataset[0]
            wfs.SetPhotonsAndRON(batch["nphotons"], batch["ron"])
            Ze = batch["opd"].flatten(start_dim=-2) @ z_inv
            frames = framePreprocessor.ProcessFrame(wfs(batch["opd"], batch["pupil"]), False)
            for name, m in models.items():
                num[name] = num[name] + ((m(frames) - Ze) ** 2).sum(0)
            den = den + (Ze ** 2).sum(0)
    for m in models.values():
        m.train()
    return {name: (num[name] / den).cpu().numpy() for name in models}


errors = per_mode_error(models)

fig, ax = plt.subplots(figsize=(10, 4))
for name, err in errors.items():
    ax.plot(err, lw=2, color=COLORS[name], label=name)
ax.axhline(1.0, color="0.3", lw=1, ls=":")
ax.set_xlabel("mode index i  (Noll j = i + 2)")
ax.set_ylabel("⟨(ẑ − z)²⟩ / ⟨z²⟩")
ax.grid(alpha=0.3)
ax.legend(fontsize="small", loc="center left", bbox_to_anchor=(1.01, 0.5))
fig.tight_layout()
plt.show()

bands = {"tip/tilt": slice(0, 2), "2-8": slice(2, 9), "9-29": slice(9, 30), "30-63": slice(30, None), "all": slice(None)}
print(f"{'variant':<20}" + "".join(f"{b:>10}" for b in bands))
for name, err in errors.items():
    print(f"{name:<20}" + "".join(f"{err[s].mean():>10.3f}" for s in bands.values()))

The new layers can be inspected with the same tools as before; for example, the gradient flow of the variant with all four changes, and the learned front conv compared with its Pyramid initialization:

In [ ]:
diags["all four"].plot_gradient_flow(source="weight");

front = models["all four"].front.weight.detach().cpu()[:, :, 0, 0]
print("front conv now (rows: flux, x-slope, y-slope, cross):")
print(front)
print("change from the Pyramid initialization:",
      float((front - pyramid_combinations(wfs.pupil_centers)).norm()))